**Student template - TODOs are intentional.**

# COMP9312 26T3 - Q2: connected k-core index

Implement `KCoreIndex.build()`, `query(k, v)` and `max_k(u, v)`. The constructor receives `n` and a simple undirected edge list; you may change internal representations while preserving signatures.

- `build()` preprocesses once and returns `self`.
- `query(k, v)` returns an ascending list of distinct vertex IDs; `[]` if absent.
- `k` is a non-negative integer. At `k == 0`, return the ordinary connected component.
- `max_k(u, v)` returns the largest shared threshold; `-1` if disconnected; the core number if `u == v` (0 for an isolated vertex).
- Inputs use IDs `0..n-1`. If `n == 0`, edges are empty and no vertex queries occur. Calls follow `build()`.

Use only the Python Standard Library. Put helpers/imports before the local-test cell. Read the accompanying Q2 specification and Project Summary.


In [ ]:
class KCoreIndex:
    def __init__(self, n, edges):
        self.n = n
        self.adj = [[] for _ in range(n)]
        for u, v in edges:
            self.adj[u].append(v)
            self.adj[v].append(u)

    def build(self):
        # TODO: construct your index here.
        return self

    def query(self, k, v):
        # TODO: return the component in ascending vertex-ID order.
        return []

    def max_k(self, u, v):
        # TODO: return the maximum shared core threshold.
        return -1


## Local tests
Run after implementing. Do not change this cell. Timings are informational, not fixed grading limits. The test generator and full-output digests are deterministic.

In [ ]:
# LOCAL TESTS - keep this marker and the tests unchanged.
"""Deterministic fixtures and independent oracles; Standard Library only."""
import hashlib
import heapq
from collections import deque


class FixedRandom:
    """Explicit xorshift32 test generator, independent of random module versions."""
    def __init__(self, seed):
        self.state = seed & 0xffffffff or 1

    def below(self, limit):
        x = self.state
        x ^= (x << 13) & 0xffffffff
        x ^= x >> 17
        x ^= (x << 5) & 0xffffffff
        self.state = x & 0xffffffff
        return self.state % limit


def graph(n, m, seed=1):
    assert 0 <= m <= n * (n - 1) // 2
    rng, seen, edges = FixedRandom(seed), set(), []
    while len(edges) < m:
        u, v = rng.below(n), rng.below(n)
        if u == v:
            continue
        edge = (min(u, v), max(u, v))
        if edge not in seen:
            seen.add(edge)
            edges.append(edge)  # Never enumerate the set.
    return edges


def pairs(n, count, seed=2):
    rng = FixedRandom(seed)
    return [(rng.below(n), rng.below(n)) for _ in range(count)]


def query_pairs(n, count, seed=3):
    rng = FixedRandom(seed)
    return [(rng.below(13), rng.below(n)) for _ in range(count)]


def add_digest(digest, values):
    digest.update((",".join(map(str, values)) + "\n").encode("ascii"))


def digest_rows(rows):
    digest = hashlib.sha256()
    for row in rows:
        add_digest(digest, row)
    return digest.hexdigest()


def brute_q1(n, edges, queries):
    answers = []
    for a, b in queries:
        if a == b:
            answers.append(0)
            continue
        adj, answer = [set() for _ in range(n)], -1
        for t, (u, v) in enumerate(edges, 1):
            adj[u].add(v)
            adj[v].add(u)
            seen, pending = {a}, [a]
            while pending:
                u = pending.pop()
                for v in adj[u] - seen:
                    seen.add(v)
                    pending.append(v)
            if b in seen:
                answer = t
                break
        answers.append(answer)
    return answers


def brute_q2(n, edges, k, v):
    adj = [set() for _ in range(n)]
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)
    alive, degree = [True] * n, [len(row) for row in adj]
    pending = deque(x for x in range(n) if degree[x] < k)
    while pending:
        x = pending.popleft()
        if not alive[x]:
            continue
        alive[x] = False
        for y in adj[x]:
            if alive[y]:
                degree[y] -= 1
                if degree[y] < k:
                    pending.append(y)
    if not alive[v]:
        return []
    seen, pending = {v}, [v]
    while pending:
        x = pending.pop()
        for y in adj[x]:
            if alive[y] and y not in seen:
                seen.add(y)
                pending.append(y)
    return sorted(seen)


def brute_max(n, edges, u, v):
    answer = -1
    for k in range(n + 1):
        if v in brute_q2(n, edges, k, u):
            answer = k
        else:
            break
    return answer



import time
edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3),(3,4),(4,5),(3,5),(5,6),(6,7),(6,8),(6,9),(7,8),(7,9),(8,9),(9,10)]
idx = KCoreIndex(11,edges).build()
assert idx.query(1,0)==list(range(11)) and idx.query(2,4)==list(range(10))
assert idx.query(3,3)==[0,1,2,3] and idx.query(3,7)==[6,7,8,9]
assert idx.query(4,0)==[] and idx.query(2,10)==[]
assert [idx.max_k(u,v) for u,v in [(0,7),(0,4),(4,10),(1,2)]]==[2,2,1,3]
idx = KCoreIndex(5,[(0,1),(1,2),(0,2)]).build()
assert idx.query(0,4)==[4] and idx.query(1,4)==[]
assert idx.query(0,0)==[0,1,2] and idx.query(99,0)==[]
assert idx.max_k(4,4)==0 and idx.max_k(3,4)==-1 and idx.max_k(0,0)==2
KCoreIndex(0,[]).build()
rng = FixedRandom(9312)
for trial in range(150):
    n = 2 + rng.below(11)
    E = graph(n,rng.below(min(24,n*(n-1)//2)+1),trial+100)
    idx = KCoreIndex(n,E).build()
    for _ in range(6):
        k,v = rng.below(6),rng.below(n)
        assert idx.query(k,v)==brute_q2(n,E,k,v), (n,E,k,v)
    for u,v in pairs(n,6,trial+1000)+[(0,0)]:
        assert idx.max_k(u,v)==brute_max(n,E,u,v), (n,E,u,v)
print("Q2 figure, boundary cases and 150 oracle trials passed")
n = 10000
E = graph(n,80000,1)
start = time.perf_counter()
idx = KCoreIndex(n,E).build()
build_time = time.perf_counter()-start
digest = hashlib.sha256()
start = time.perf_counter()
for k,v in query_pairs(n,2000,3):
    answer = idx.query(k,v)
    assert type(answer) is list
    assert all(type(value) is int for value in answer)
    add_digest(digest,answer)
query_time = time.perf_counter()-start
assert digest.hexdigest()=='f303322c378ce85a9afee5ad04ba6735c36dddb7ff95a360ebfea2cd65b35b88', "Q2 full-membership digest mismatch"
start = time.perf_counter()
answers = [idx.max_k(u,v) for u,v in pairs(n,20000,4)]
max_time = time.perf_counter()-start
assert all(type(value) is int for value in answers)
assert digest_rows([answers])=='39ce2689f77559bdbf3079ee376211c4dfc95a3e5c52b00f7fb22fcb6c88f549', "Q2 max_k digest mismatch"
print("Q2 scale passed; build %.3f s, queries+digest %.3f s, max_k %.3f s" % (build_time,query_time,max_time))
